In [1]:
# Notebook to map ares in Schaefer parcellation which are persistent in the standard experiment
# to areas in the Glasser parcellation and save brain plots of the Schaefer areas
# which we can compare to see if the locations match the location on the Glasser parcellation

In [2]:
import numpy as np
import nibabel.freesurfer as fs
from nilearn import plotting, datasets
import pandas as pd
import matplotlib.pyplot as plt
import os

In [4]:
persistent_areas = np.load('../models/model_pet_nmda/results/main_task/persistent_areas_G083_alpha134.npy')
glasser_schaefer_mapping = pd.read_csv('results/literature_map_glasser_schaefer/mapping_glasser_to_schaefer200.csv')
schaefer_area_names = pd.read_csv('../data/processed/schaefer200_LH_area_names.csv')

In [5]:
schaefer_names = ['7Networks_LH_' + str(a) for a in persistent_areas]
df = pd.DataFrame({
    'SchaeferName': schaefer_names
})

In [6]:
# df merge: it takes each row in df and reads the value in SchaeferName
# it searches glasser_schaefer_mapping for a row where Schaefer Region Maximum Overlap has the same value
# when it finds a match, it attaches the columns from the matching row to the row in df
# because how='left' is used, every row in df stays in the result, even if no match exists
result = df.merge(
    glasser_schaefer_mapping,
    left_on='SchaeferName',
    right_on='Schaefer Region Maximum Overlap',
    how='left'
)

In [7]:
# 'result' has multiple rows with the same SchaeferName and different Glasser Region
# I combine these into a single dataframe with one unique SchaeferName per row and the corresponding
# Glasser regions (could be 0, 1 or more than 1)
df_combined_glasser_names = (
    result
    .groupby("SchaeferName")["Glasser Region"]
    .apply(lambda x: ", ".join(x.dropna().astype(str).unique()))
    .reset_index()
)

In [8]:
df_combined_glasser_names.to_csv('results/literature_map_glasser_schaefer/schaefer_glasser_mapping_persistent_areas.csv', index=False)
df_combined_glasser_names.to_excel("results/literature_map_glasser_schaefer/schaefer_glasser_mapping_persistent_areas.xlsx", index=False)

In [9]:
df_combined_glasser_names

,SchaeferName,Glasser Region
0,7Networks_LH_Cont_Cing_2,L_33pr_ROI
1,7Networks_LH_Cont_OFC_1,"L_47m_ROI, L_11l_ROI"
2,7Networks_LH_Cont_PFCl_1,"L_a47r_ROI, L_p47r_ROI"
3,7Networks_LH_Cont_PFCl_2,
4,7Networks_LH_Cont_PFCl_3,"L_IFSa_ROI, L_a9-46v_ROI"
5,7Networks_LH_Cont_Par_1,L_PFm_ROI
6,7Networks_LH_Cont_Temp_1,L_TE1p_ROI
7,7Networks_LH_Default_PFC_1,"L_47s_ROI, L_AVI_ROI, L_AAIC_ROI"
8,7Networks_LH_Default_PFC_10,L_8BM_ROI
9,7Networks_LH_Default_PFC_11,L_8Av_ROI


In [ ]:
# Method to extract index of a specific area 
specific_area = '7Networks_LH_DorsAttn_Post_9'
index_specific_area = schaefer_area_names.index[schaefer_area_names['Area_name'] == specific_area].tolist()[0]

In [11]:
# Script which iterates through the persistent areas, extracts relevant Glasser label, plots the area
# on the brain as Schaefer in a separate folder for each area
# then I can manually look at the areas (40 in total) and look at the Glasser parcellation in ../data/raw/Glasser_map/glasser_with_labels.jpg
# CAREFUL: the indices in the files we use (e.g. area names) don't correspond to the indices in region names or labels 
# here, they have to be + 1, because the labels here also have background at index 0
labels_schaefer, ctab_schaefer, region_names_schaefer = fs.read_annot('../data/raw/brain_annotations/Schaefer/fsaverage/atl-Schaefer2018_space-fsaverage_hemi-L_desc-200Parcels7Networks_deterministic.annot')
region_names_schaefer = [r.decode('utf-8') for r in region_names_schaefer]
for persist_area in df_combined_glasser_names['SchaeferName']:
    index_specific_area = schaefer_area_names.index[schaefer_area_names['Area_name'] == persist_area].tolist()[0]
    path = f'results/literature_map_glasser_schaefer/{persist_area}'
    os.makedirs(path, exist_ok =True)
    mapped_data_schaefer_persist_area = np.zeros_like(labels_schaefer, dtype=float) # Define a map with the same number of vertices as the annotation file
    lesion_indices_less_important = []
    lesion_indices_important = [index_specific_area + 1]
    for i, region in enumerate(region_names_schaefer):
        region_idx = i
        if i == 0:
            mapped_data_schaefer_persist_area[labels_schaefer == region_idx] = None
        else:
            if i in lesion_indices_less_important:
                mapped_data_schaefer_persist_area[labels_schaefer == region_idx] = 1 # First index in region_names is background
            if i in lesion_indices_important:
                print(region)
                mapped_data_schaefer_persist_area[labels_schaefer == region_idx] = 2 # First index in region_names is background
    fsaverage = datasets.fetch_surf_fsaverage('fsaverage') # Extract surface to plot on
    vmin = np.nanmin(mapped_data_schaefer_persist_area) # Helps with plotting colors
    vmax = np.nanmax(mapped_data_schaefer_persist_area) # Helps with plotting colors
    fs = datasets.fetch_surf_fsaverage('fsaverage')
    for view in ['lateral','medial']:
        fig = plt.figure(constrained_layout=True, figsize=(6,6))
        ax = fig.add_subplot(1,1,1, projection='3d')
        plotting.plot_surf_stat_map(
            fs.infl_left, mapped_data_schaefer_persist_area, fs.sulc_left,
            hemi='left', view=view,
            cmap='plasma', colorbar=True,
            symmetric_cbar=False,
            vmin=vmin, vmax=vmax,
            axes=ax, figure=fig,
            engine='matplotlib', avg_method='median',
            title=f'{persist_area}'
        )
        for ext in ['png']:
            dpi = 150 if ext in ['png'] else None
            fig.savefig(f'results/literature_map_glasser_schaefer/{persist_area}/schaefer_persist_{persist_area}_{view}.{ext}', dpi=dpi, bbox_inches='tight')
        plt.close()

7Networks_LH_Cont_Cing_2
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
7Networks_LH_Cont_OFC_1
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
7Networks_LH_Cont_PFCl_1
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
7Networks_LH_Cont_PFCl_2
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
7Networks_LH_Cont_PFCl_3
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
7Networks_LH_Cont_Par_1
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage
[get_dataset_dir] Dataset found in /home/rad216/nilearn_data/fsaverage


: 